# Spark Tune - Databricks ML Pipeline Demo

End-to-end ML pipeline reading data from a Databricks catalog and demonstrating:

1. **ydata-profiling** - Data exploration & quality profiling

In [0]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import warnings
warnings.filterwarnings('ignore')

In [0]:
# Uncomment to install dependencies if needed
# !pip install -r /Workspace/Users/yadvendra@aidetic.in/spark_beyond/requirements.txt

---
## 1. Load Data from Databricks Catalog

In [0]:
from backend.core.utils import process_col_names

# Read from Databricks Unity Catalog volume
# df = spark.read.csv(
#     "/Volumes/aidetic_databricks/default/credit_card_transactions/credit_card_transactions.csv",
#     header=True,
#     inferSchema=True
# )

# df = df.drop("Unnamed: 0")

CATALOG_NAME = "aidetic_databricks"
SCHEMA_NAME = "default"

# # Transaction Data
# TABLE_NAME = "credit_card_transactions"

# # Bank Customers
# TABLE_NAME = "hdfc_demo_bank_customers"

# Bank Customers
TABLE_NAME = "hdfc_demo_bank_transactions"

credit_card_transactions_table_name = f"{CATALOG_NAME}.{SCHEMA_NAME}.{TABLE_NAME}"

df = spark.read.table(credit_card_transactions_table_name)

print(f"Dataset shape: {df.count():,} rows x {len(df.columns)} columns")
df.printSchema()

In [0]:
display(df.limit(5))

---
## 3. YData Profiling - Data Exploration

Generate a comprehensive data profile using ydata-profiling.
The `quick_profile` function auto-samples large datasets to avoid memory issues.

In [0]:
from ydata_profiling import ProfileReport

# Full profiling report (saved to HTML)
profile = ProfileReport(
    df.sample(fraction=0.01, seed=42).toPandas(),
    title="Credit Card Transactions - Profiling Report",
    explorative=True
)

profile.to_file(f"{TABLE_NAME}_data_profiling_report.html")
print("Full profiling report saved to 'data_profiling_report.html'")

In [0]:
from backend.core.profiling.ydata_profiler import quick_profile

# Quick profile for summary stats & alerts
quick_stats = quick_profile(df, max_rows=10000)

summary = quick_stats['summary']
print("QUICK PROFILE SUMMARY:")
print(f"  Rows: {summary.get('n_rows', 0):,}")
print(f"  Columns: {summary.get('n_columns', 0)}")
print(f"  Missing Cells: {summary.get('missing_cells_pct', 0):.2f}%")
print(f"  Duplicate Rows: {summary.get('duplicate_rows_pct', 0):.2f}%")

In [0]:
# Display alerts from profiling
print("DATA ALERTS:")
print("-" * 40)
if quick_stats['alerts']:
    for alert in quick_stats['alerts'][:15]:
        print(f"  - {alert['column']}: {alert['type']}")
else:
    print("  No alerts detected.")

print("\nPROFILING RECOMMENDATIONS:")
print("-" * 40)
if quick_stats['recommendations']:
    for rec in quick_stats['recommendations'][:10]:
        print(f"  [{rec['priority'].upper()}] {rec['column']}: {rec['action']}")
else:
    print("  No recommendations.")

---
## 4. Time-Series Detection

Automatically identify temporal structure in the dataset before running tsfresh.

In [0]:
from backend.core.discovery import Problem, SchemaChecks

problem = Problem(
    # target="is_fraud", #For credit card transactions
    # target="responded", # For bank customers
    target = "is_flagged", # For bank transactions
    type="classification",
    desired_result=1,
    # date_column="trans_date_trans_time" #For credit card transactions
    # date_column="contact_timestamp" # For bank customers
    date_column="transaction_date"
)

schema_checker = SchemaChecks(dataframe=df, problem=problem)
schema_info = schema_checker.check()

print(f"Problem Type: {problem.type}")
print(f"Target Column: {problem.target}")
print(f"Desired Result: {problem.desired_result}")
print(f"\nSchema Summary:")
print(f"  Categorical columns: {len(schema_info['categorical'])}")
print(f"  Numerical columns: {len(schema_info['numerical'])}")
print(f"  Boolean columns: {len(schema_info['boolean'])}")

In [0]:
from backend.core.utils.time_series_detector import detect_time_series_structure

ts_info = detect_time_series_structure(df, schema_checker)

print("TIME SERIES DETECTION RESULTS:")
print("-" * 40)
print(f"  Is Time Series: {ts_info.is_time_series}")
print(f"  Time Column: {ts_info.time_column or 'N/A'}")
print(f"  Frequency: {ts_info.frequency.value if ts_info.frequency else 'N/A'}")
print(f"  Entity Columns: {ts_info.entity_columns or 'N/A'}")

if ts_info.recommended_features:
    print("\nRecommended Time-Series Features:")
    for feature in ts_info.recommended_features:
        print(f"    - {feature}")